[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# Reading and Writing Files


## What you will be able to do

Write a GeoDataFrame to GeoJSON, GeoPackage and Shapefile, read it back with its CRS, choose a layer
from a file that holds several, and read only the rows and columns you need.

## The idea

### The problem

The forecast office sends its district outlines to a mapping team, the mapping team sends back a file
of coastline features, and a partner agency publishes its stations for download. Every one of those
exchanges is a file, and a spatial file has to carry more than a CSV does. It holds the shapes, the
attribute columns beside them, and the coordinate reference system that says what the numbers in the
shapes mean. A district outline in meters, saved without its CRS, reaches the next person as a list of
numbers in the millions with nothing to say they are meters east and north in UTM zone 33N.

There are many formats, and each one keeps a different set of these things. A GeoJSON file is text
that any program can open. A GeoPackage is a single database file that can hold several tables. A
Shapefile is the oldest of the three and still common, and it limits column names to ten characters.
Knowing what each one keeps decides which to send and what to check when one arrives.

### A formal definition

> A **vector data file** stores **features**, each a geometry with a row of attribute values, in one or
> more **layers**, each a table whose features share a geometry type and a CRS. GeoPandas reads a layer
> into a GeoDataFrame with `read_file` and writes one with `to_file`, through the **pyogrio** library,
> which uses GDAL, the open-source library that reads and writes most spatial formats.

### Why it works that way

`read_file` and `to_file` hand the work to GDAL, so the format is chosen by the file's extension and
GeoPandas never needs code of its own for each format. What comes back depends on what the format
stores. A GeoPackage keeps every layer's CRS and its column types, so a table written and read again
comes back as it went in. GeoJSON's standard expects longitude and latitude in WGS 84, and GDAL also
writes a `crs` member that names the layer's CRS, as the first worked example shows. A Shapefile is
several files that travel together, and it cuts a column name longer than ten characters down to ten.

Reading is where files differ most from a CSV. A GeoPackage can hold a large layer, and `read_file`
can read part of it, the features inside a box or a shape, the rows that match a condition, or a few
columns, so that a program never loads what it does not need.

### Where this shows up

The country outlines in this guide began as a file, which Setup reads from GeoJSON. Government mapping agencies, statistics offices and open-data portals publish in these
formats, and the **A Station Atlas** notebook writes its results to a GeoPackage with two layers.

### What this notebook covers

- GeoJSON, as text you can read, and what comes back from it
- a GeoPackage holding several layers, each with its own CRS
- reading a file's description before reading its features
- reading part of a layer by box, by condition, by shape and by column
- a Shapefile and the files it is made of
- a CSV with its geometry written as text
- three mistakes made with files

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import geopandas as gpd
from shapely.geometry import Point

towns = gpd.GeoDataFrame({"town": ["Oslo", "Bergen"]},
                         geometry=[Point(10.75, 59.91), Point(5.32, 60.39)], crs=4326)
towns.to_file("towns.gpkg", layer="towns")
back = gpd.read_file("towns.gpkg", layer="towns")
print(back)
print(back.crs)
```

```
     town             geometry
0    Oslo  POINT (10.75 59.91)
1  Bergen   POINT (5.32 60.39)
EPSG:4326
```

Two points go into a GeoPackage as a layer named `towns`, and `read_file` brings back the same rows,
the same geometry and the same CRS.

## Setup

Four libraries for the tables and the geometry, pyogrio for looking inside files, and three tools from
the standard library for the scratch folder and the warnings.

- `shutil` removes the scratch folder at the end of Common errors
- `warnings` catches the warnings that reading and writing give, so that a cell can print them
- `Path` names the files, and finds the guide's copy of the country outlines when the notebook runs
  beside it
- `numpy` and `pandas` build the made-up register
- `geopandas` reads and writes every file, and the cell prints its version
- `shapely` builds the made-up districts
- `pyogrio` lists a file's layers and describes a layer without reading it, and the cell prints its
  version and the version of GDAL inside it

`countries` holds the outlines of the Nordic countries. Made with Natural Earth. The outlines come
from Natural Earth's 1:50m map of the world's countries, which is in the public domain.

`stations` is the register of 300 weather stations as a GeoDataFrame in latitude and longitude, and
`named` is the five named stations. `districts` is the eight forecast districts, in meters in UTM zone
33N. The stations and the districts are made up, drawn from a fixed seed so that they are the same on
every run. Every file this notebook writes goes into a folder named `scratch`, which the last cell
removes.

In [1]:
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pyogrio

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


weather_stations = pd.DataFrame({
    "name": ["Bergen", "Oslo", "Svalbard", "Tromso", "Kirkenes"],
    "latitude": [60.39, 59.91, 78.22, 69.65, 69.73],
    "longitude": [5.32, 10.75, 15.65, 18.96, 30.05],
})


def mainland_norway():
    """Norway's outline south of 74 degrees north, without Svalbard, in UTM zone 33N."""
    parts = countries[countries["ADMIN"] == "Norway"].to_crs(25833).explode()
    return parts[parts.geometry.bounds["maxy"] < 8_200_000].union_all()


def station_network(n=300, seed=2026):
    """A made-up register of weather stations on mainland Norway, one row each."""
    rng = np.random.default_rng(seed)
    land = mainland_norway()
    x_min, y_min, x_max, y_max = land.bounds
    east, north = [], []
    while len(east) < n:
        x, y = rng.uniform(x_min, x_max), rng.uniform(y_min, y_max)
        if land.contains(shapely.Point(x, y)):
            east.append(x)
            north.append(y)
    lonlat = gpd.GeoSeries(gpd.points_from_xy(east, north), crs=25833).to_crs(4326)
    return pd.DataFrame({
        "station": [f"NS{i:03d}" for i in range(1, n + 1)],
        "latitude": lonlat.y.round(4).to_numpy(), "longitude": lonlat.x.round(4).to_numpy(),
        "elevation_m": np.clip(rng.gamma(1.6, 220, n), 2, 1900).round(0).astype(int),
        "owner": rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2]),
        "installed": rng.integers(1950, 2025, n),
    })


DISTRICT_CENTERS = {"South": (8.0, 58.6), "East": (10.8, 60.5), "West": (6.0, 60.8), "Mid": (11.0, 63.6),
                    "North": (14.5, 67.0), "Far North": (19.0, 69.4), "Northeast": (25.5, 70.0),
                    "Svalbard": (16.0, 78.5)}


def forecast_districts():
    """Eight made-up forecast districts that cover Norway, in UTM zone 33N."""
    norway = countries[countries["ADMIN"] == "Norway"].to_crs(25833).geometry.iloc[0]
    centers = gpd.GeoSeries(gpd.points_from_xy(*zip(*DISTRICT_CENTERS.values())), crs=4326).to_crs(25833)
    cells = shapely.voronoi_polygons(shapely.MultiPoint(list(centers)),
                                     extend_to=shapely.box(*norway.buffer(200_000).bounds)).geoms
    names = [next(name for name, center in zip(DISTRICT_CENTERS, centers) if cell.contains(center))
             for cell in cells]
    districts = gpd.GeoDataFrame({"district": names}, geometry=[cell.intersection(norway) for cell in cells],
                                 crs=25833)
    return districts.set_index("district").loc[list(DISTRICT_CENTERS)].reset_index()


network = station_network()
stations = gpd.GeoDataFrame(
    network, geometry=gpd.points_from_xy(network["longitude"], network["latitude"]), crs=4326
)


named = gpd.GeoDataFrame(
    weather_stations,
    geometry=gpd.points_from_xy(weather_stations["longitude"], weather_stations["latitude"]),
    crs=4326,
)


districts = forecast_districts()
scratch = Path("scratch")
scratch.mkdir(exist_ok=True)
print("geopandas", gpd.__version__, "| pyogrio", pyogrio.__version__, "| GDAL", pyogrio.__gdal_version_string__,
      "|", len(stations), "made-up stations,", len(districts), "made-up districts")

geopandas 1.2.0 | pyogrio 0.13.0 | GDAL 3.12.4 | 300 made-up stations, 8 made-up districts


## Worked examples

### GeoJSON as readable text

`to_file` picks the format from the extension, so a name ending in `.geojson` writes GeoJSON.

In [2]:
stations.to_file(scratch / "stations.geojson")
lines = (scratch / "stations.geojson").read_text().splitlines()
for line in lines[:5]:
    print(line)
print(lines[5][:110], "...")

{
"type": "FeatureCollection",
"name": "stations",
"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
"features": [
{ "type": "Feature", "properties": { "station": "NS001", "latitude": 61.1793, "longitude": 10.6233, "elevation ...


The file is text. Its `crs` member names `CRS84`, which is WGS 84 longitude and latitude, the order
GeoJSON's standard uses, and every feature is a dictionary of `properties` with a `geometry` beside
it. Reading it back gives a GeoDataFrame again.

In [3]:
from_geojson = gpd.read_file(scratch / "stations.geojson")
print(type(from_geojson).__name__, "| CRS:", from_geojson.crs, "|", len(from_geojson), "rows")
print("same points:", from_geojson.geom_equals(stations).all())
print(from_geojson.dtypes)
print("same elevations:", (from_geojson["elevation_m"] == stations["elevation_m"]).all())

GeoDataFrame | CRS: EPSG:4326 | 300 rows
same points: True
station          object
latitude        float64
longitude       float64
elevation_m       int32
owner            object
installed         int32
geometry       geometry
dtype: object
same elevations: True


The 300 points and the CRS come back as they went in. The whole-number columns come back as `int32`,
where the register held `int64`, because a GeoJSON file records no column types and GDAL picks a
32-bit integer when every value fits in one. The elevations themselves are the same.

### A GeoPackage with several layers

A GeoPackage is one file that can hold several layers. `layer=` names the one to write, and writing a
second name adds a second layer to the same file.

In [4]:
atlas = scratch / "atlas.gpkg"
districts.to_file(atlas, layer="districts")
stations.to_file(atlas, layer="stations")
pyogrio.list_layers(atlas)

array([['districts', 'MultiPolygon'],
       ['stations', 'Point']], dtype=object)

`list_layers` gives each layer's name and geometry type without reading any features. Each layer keeps
its own CRS, so the districts come back in meters and the stations in degrees.

In [5]:
districts_back = gpd.read_file(atlas, layer="districts")
stations_back = gpd.read_file(atlas, layer="stations")
print("districts:", districts_back.crs, "|", len(districts_back), "rows")
print("stations: ", stations_back.crs, "|", len(stations_back), "rows")
print(stations_back.dtypes)

districts: EPSG:25833 | 8 rows
stations:  EPSG:4326 | 300 rows
station          object
latitude        float64
longitude       float64
elevation_m       int64
owner            object
installed         int64
geometry       geometry
dtype: object


The GeoPackage stores the column types too, so `elevation_m` and `installed` come back as `int64`, as
the register had them.

Without `layer=`, `read_file` reads the first layer and warns that there were others. The cell catches
the warning and prints it.

In [6]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    first = gpd.read_file(atlas)
for warning in caught:
    print(f"{warning.category.__name__}: {str(warning.message).splitlines()[0]}")
print("read:", list(first.columns), len(first), "rows")

read: ['district', 'geometry'] 8 rows


### Describing a layer before reading it

`pyogrio.read_info` describes one layer: its CRS, geometry type, number of features, column names and
the box around all its features, read from the file's own records.

In [7]:
info = pyogrio.read_info(atlas, layer="stations")
print("CRS:", info["crs"], "| geometry:", info["geometry_type"], "| features:", info["features"])
print("columns:", list(info["fields"]))
print("bounds:", [round(value, 2) for value in info["total_bounds"]])

CRS: EPSG:4326 | geometry: Point | features: 300
columns: ['station', 'latitude', 'longitude', 'elevation_m', 'owner', 'installed']
bounds: [5.05, 58.35, 30.58, 71.03]


The bounds run from 5.05 to 30.58 degrees east and from 58.35 to 71.03 north, which is mainland
Norway. Checking this before a large read tells you whether a file covers the area you need and is in
the CRS you expect.

### Reading part of a layer

`bbox` keeps the features inside a box, given as west, south, east and north in the layer's own CRS,
here degrees.

In [8]:
southwest = gpd.read_file(atlas, layer="stations", bbox=(4.5, 58.0, 8.0, 62.0))
print(len(southwest), "stations between 4.5 and 8 degrees east and 58 and 62 north")

51 stations between 4.5 and 8 degrees east and 58 and 62 north


`where` takes a condition written in SQL, the language databases use, and `columns` keeps only the
columns named, with the geometry always included.

In [9]:
airports = gpd.read_file(atlas, layer="stations", where="owner = 'airport'", columns=["station", "elevation_m"])
print(len(airports), "airport stations | columns:", list(airports.columns))
print("in the register:", (stations["owner"] == "airport").sum())

33 airport stations | columns: ['station', 'elevation_m', 'geometry']
in the register: 33


`mask` keeps the features inside a shape. Given a GeoDataFrame, `read_file` converts it to the
layer's CRS first, so the West district, in meters, can filter the stations, in degrees.

In [10]:
west = districts[districts["district"] == "West"]
in_west = gpd.read_file(atlas, layer="stations", mask=west)
by_join = gpd.sjoin(stations.to_crs(districts.crs), west)
print(len(in_west), "stations read inside West |", len(by_join), "by a spatial join")

43 stations read inside West | 43 by a spatial join


Both give 43, the West count from the **Why GeoPandas** notebook. The filter runs while the file is
read, so the other 257 stations never reach the GeoDataFrame.

### A Shapefile and the files it is made of

A name ending in `.shp` writes a Shapefile. This cell writes three columns whose names are short
enough for the format.

In [11]:
stations[["station", "owner", "installed", "geometry"]].to_file(scratch / "stations.shp")
print(sorted(path.name for path in scratch.glob("stations.*")))
print((scratch / "stations.prj").read_text()[:44], "...")

['stations.cpg', 'stations.dbf', 'stations.geojson', 'stations.prj', 'stations.shp', 'stations.shx']
GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPH ...


This Shapefile is five files with the same name. `.shp` holds the shapes, `.dbf` the columns, `.shx` an
index into the shapes, `.prj` the CRS written as text, and `.cpg` the character encoding of the
columns. Send them together, since GDAL reads the CRS from the `.prj`, as the next cell shows. The
`stations.geojson` above is a separate file that happens to share the name.

In [12]:
from_shapefile = gpd.read_file(scratch / "stations.shp")
print(from_shapefile.crs, "|", list(from_shapefile.columns))
(scratch / "stations.prj").unlink()
print("without .prj:", gpd.read_file(scratch / "stations.shp").crs)

EPSG:4326 | ['station', 'owner', 'installed', 'geometry']
without .prj: None


### A CSV with its geometry as text

`to_csv` is pandas' method, and it writes each geometry as WKT, the text form of a shape. The CSV has
no place for a CRS.

In [13]:
stations.to_csv(scratch / "stations.csv", index=False)
print((scratch / "stations.csv").read_text().splitlines()[1])
plain = pd.read_csv(scratch / "stations.csv")
print(type(plain).__name__, "| geometry holds", type(plain["geometry"].iloc[0]).__name__)

NS001,61.1793,10.6233,191,met,2019,POINT (10.6233 61.1793)
DataFrame | geometry holds str


`read_csv` gives a plain DataFrame whose `geometry` column holds strings. `GeoSeries.from_wkt` turns
the strings back into shapes, and the CRS has to be given again, since the file never held it.

In [14]:
rebuilt = gpd.GeoDataFrame(plain, geometry=gpd.GeoSeries.from_wkt(plain["geometry"]), crs=4326)
print(type(rebuilt).__name__, "| CRS:", rebuilt.crs, "| same points:", rebuilt.geom_equals(stations).all())

GeoDataFrame | CRS: EPSG:4326 | same points: True


## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/05-reading-and-writing-files-solutions.ipynb).

**1.** Write `districts` to a GeoJSON file in `scratch`, read it back, and print its CRS and the first four lines of the file.

In [15]:
# your code here

**2.** Add the five named stations to `atlas.gpkg` as a layer named `named`, then list the file's layers.

In [16]:
# your code here

**3.** Read only the stations owned by `road` from the `stations` layer of `atlas.gpkg`, keeping the `station` and `elevation_m` columns, and print how many there are and their mean elevation.

In [17]:
# your code here

**4.** Read the stations inside the Mid district with `mask`, and check the count against a spatial join.

In [18]:
# your code here

**5.** Write the stations to a Shapefile with `elevation_m` renamed so that no column name is cut short, read it back, and print its columns.

In [19]:
# your code here

## Common errors

### `DataLayerError: Layer 'station' could not be opened`

The cell writes the register to a GeoPackage as a layer named `stations`, and then asks for
`station`.

In [20]:
stations.to_file(scratch / "layers.gpkg", layer="stations")
gpd.read_file(scratch / "layers.gpkg", layer="station")

DataLayerError: Layer 'station' could not be opened

A file can hold several layers, and `read_file` opens only the one named exactly. `pyogrio.list_layers`
prints the names a file holds, so check it and copy the name from it.

In [21]:
print(pyogrio.list_layers(scratch / "layers.gpkg")[:, 0])
print(len(gpd.read_file(scratch / "layers.gpkg", layer="stations")), "stations")

['stations']
300 stations


### `UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.`

The whole register goes into a Shapefile, `elevation_m` and all.

In [22]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    warnings.simplefilter("always", RuntimeWarning)
    stations.to_file(scratch / "register.shp")
for warning in caught:
    print(f"{warning.category.__name__}: {str(warning.message).splitlines()[0]}")
print(list(gpd.read_file(scratch / "register.shp").columns))

['station', 'latitude', 'longitude', 'elevation_', 'owner', 'installed', 'geometry']


A Shapefile keeps its columns in a `.dbf` file, a format that allows names of ten characters at most.
GDAL cuts `elevation_m`, eleven characters, to `elevation_`, says so in a second warning, and writes
the file, so code that later asks for `elevation_m` will not find it. Rename long
columns before writing, as task 5 asks, or write a GeoPackage, which keeps the names.

### No error, and a layer that lost 290 stations

The cell writes the register to a GeoPackage of its own, then saves a sample of ten stations under the
same layer name.

In [23]:
sample_file = scratch / "sample.gpkg"
stations.to_file(sample_file, layer="stations")
stations.head(10).to_file(sample_file, layer="stations")
print(len(gpd.read_file(sample_file, layer="stations")), "stations in the layer")

10 stations in the layer


Writing to a layer that already exists replaces it, so the 300 stations are gone and nothing warned.
Give the sample a layer name of its own, or pass `mode="a"` to add its rows to the end of the
layer.

In [24]:
stations.to_file(sample_file, layer="stations")
stations.head(10).to_file(sample_file, layer="sample")
for name, kind in pyogrio.list_layers(sample_file):
    print(name, len(gpd.read_file(sample_file, layer=name)))
stations.head(10).to_file(sample_file, layer="stations", mode="a")
print("stations after appending:", len(gpd.read_file(sample_file, layer="stations")))

stations 300
sample 10
stations after appending: 310


The scratch folder and every file in it can go now.

In [25]:
shutil.rmtree(scratch)
print("scratch folder still there:", scratch.exists())

scratch folder still there: False


## Recap

- You can write a GeoDataFrame to GeoJSON, GeoPackage or Shapefile with `to_file`, and read it back
  with `read_file`, CRS included.
- You can keep several layers in one GeoPackage, list them with `pyogrio.list_layers`, and read the
  one you name.
- You can describe a layer with `pyogrio.read_info` before reading it.
- You can read only the features inside a box or a shape, the rows that match a condition, and the
  columns you name.
- You can rebuild a GeoDataFrame from a CSV whose geometry was written as text.

## What is next

The **Measuring Distance and Area** notebook measures lengths, areas and distances, in a projected CRS
and on the ellipsoid, and shows what goes wrong when they are measured in degrees.

---

&#8592; **Previous:** [Coordinate Reference Systems](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/04-coordinate-reference-systems.ipynb)  &nbsp;·&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Measuring Distance and Area](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/06-measuring-distance-and-area.ipynb) &#8594;
